# 课程总结与实践

恭喜你完成了 CANN 基础知识课程的全部 4 节课！本节将把四节课的知识串联成一张完整地图，并通过三道难度递进的实践题检验学习成果。

| 课程 | 核心问题 | 一句话回答 |
|:---:|:---|:---|
| ① 人工智能基础 | AI 在算什么？ | 模型 = 参数化函数 $y=f(x;\theta)$，训练求 $\theta$，推理用 $\theta$，计算图把算子串成网络 |
| ② 什么是 NPU | 硬件怎么算？ | DaVinci 架构，AI Core = Cube + Vector + Scalar，多核并行 + Tiling 切分 |
| ③ 什么是 CANN | 软件怎么管？ | 翻译官 + 调度员 + 工具箱，从框架适配到驱动，九层组件协同 |
| ④ Hello World | 怎么上手？ | `import torch_npu` → `.npu()` → 自动执行 → `.cpu()` 验证 |

---
## 一、知识全景图：从一行代码到一颗芯片

当你写下 `output = torch.matmul(a.npu(), b.npu())` 这一行代码时，数据经历了以下完整旅程：

```
你的 Python 代码                    ← ① AI 概念：模型、算子、张量、计算图
  │
  │  torch_npu 框架适配
  ▼
CANN 算子库 MatMul                  ← ③ CANN：算子映射
  │
  │  GE 图引擎优化（融合/内存/流水）
  ▼
毕昇编译器 → NPU Kernel 指令        ← ③ CANN：编译
  │
  │  Runtime 调度（分配内存/创建流/下发）
  ▼
Driver → AI Core 执行               ← ② NPU：DaVinci 架构
  │
  │  Cube 单元矩阵乘 + Vector 单元逐元素 + 多核并行
  ▼
结果在 Device 内存 → .cpu() 搬回 Host ← ④ Hello World：搬回验证
```

**四节课的环环相扣**：
- 第 1 课回答`算什么`——AI 的数学本质与计算图抽象
- 第 2 课回答`用什么算`——NPU 硬件架构与计算单元
- 第 3 课回答`怎么管`——CANN 软件栈从框架到驱动的完整链路
- 第 4 课回答`怎么用`——5 行代码跑通第一个 NPU 程序

> ⚠️ **重要区分：算子调用 vs 算子开发**
>
> 本课程中通过 `torch_npu`（`import torch_npu` → `.npu()` → `torch.matmul` 等）使用的是**算子调用**——调用 CANN 预置的高性能算子，无需编写底层代码。
>
> 如果 CANN 算子库中没有你需要的算子，则需要用 **Ascend C** 进行**算子开发**——用 C/C++ 编写 Kernel 函数，手动管理切分、搬运、同步，编译后生成 NPU 可执行指令。这是后续教程的内容。
>
| | 算子调用（torch_npu） | 算子开发（Ascend C） |
|:---|:---|:---|
| 做什么 | 调用 CANN 内置算子 | 编写自定义算子 Kernel |
| 写什么 | Python：`torch.matmul(a, b)` | C/C++：`__aicore__ void kernel(...)` |
| 谁管切分/同步 | CANN 框架自动管 | 开发者手动管 |
| 适合场景 | 模型开发/训练/推理 | CANN 未覆盖的算子 / 极致性能优化 |

---
## 二、核心概念速查表

| 领域 | 概念 | 一句话 |
|:---|:---|:---|
| **AI** | 模型 | 参数化函数 $y=f(x;\theta)$，训练求参数，推理用参数 |
| **AI** | 计算图 | 节点=算子，边=数据流动 |
| **AI** | 算子 | 计算图中的最小计算单元（Add、MatMul、Conv…） |
| **AI** | 张量 | 算子加工的数据，由 shape 和 dtype 描述 |
| **NPU** | DaVinci | 昇腾 NPU 的核心架构名称 |
| **NPU** | AI Core | NPU 的计算核心 = 计算单元 + 存储系统 + 控制单元 |
| **NPU** | Cube/Vector/Scalar | 矩阵/向量/标量计算单元，分别对应不同粒度的运算 |
| **NPU** | Host/Device | CPU 负责`发任务传数据收结果`，NPU 负责`拼命算` |
| **NPU** | Tiling | 把大数据切分成小块分给多个核并行处理 |
| **CANN** | 框架适配 | torch_npu 让 PyTorch 代码一行改动跑在 NPU 上（**算子调用**，非算子开发） |
| **CANN** | 算子库 | ops-math/nn/cv/transformer 四大子库，预置高性能算子 |
| **CANN** | GE 图引擎 | 图编译（融合/内存/流水）+ 图执行（计算下沉） |
| **CANN** | Ascend C | 自定义算子开发语言，C/C++ 标准规范 |
| **CANN** | 毕昇编译器 | 把 Ascend C 源码编译成 NPU 可执行指令 |
| **CANN** | Runtime | 设备/内存/流/事件管理，NPU 的`管家` |

---
## 三、CANN 与 CUDA 速查对照

如果你有 GPU/CUDA 经验，这张表帮你快速建立映射：

| 维度 | CANN（昇腾） | CUDA（NVIDIA） |
|:---|:---|:---|
| 框架适配 | torch_npu | torch.cuda |
| 算子库 | ops-math / ops-nn / ops-cv / ops-transformer | cuDNN / cuBLAS / cuSPARSE |
| 通信库 | HCCL + HIXL | NCCL |
| 图引擎 | GE | TensorRT |
| 编程语言 | Ascend C | CUDA C++ |
| 编译器 | 毕昇编译器 | NVCC |
| 运行时 | ACL Runtime | CUDA Runtime |
| 设备切换 | `.npu()` | `.cuda()` |
| 架构名 | DaVinci (Cube+Vector+Scalar) | SM (CUDA Core+Tensor Core) |

---
## 四、实践题

下面三道实践题难度递进，请在代码单元中完成作答后运行验证。

| 题号 | 难度 | 考察点 | 目标 |
|:---:|:---:|:---|:---|
| 第 1 题 | ★☆☆ 基础 | 张量创建、设备搬运、逐元素运算 | 在 NPU 上完成 $z = x^2 + y$ 并验证 |
| 第 2 题 | ★★☆ 进阶 | 矩阵乘法、性能对比、数据类型 | 对比 CPU/NPU 在不同规模下的矩阵乘法耗时 |
| 第 3 题 | ★★★ 挑战 | Batch 运算、算子调用开销、性能分析 | 对比 `torch.bmm` 与循环 `torch.matmul` 的性能差异，观察 batch 大小的影响 |

### 第 1 题（基础 ★☆☆）：在 NPU 上计算 $z = x^2 + y$

**要求**：
1. 创建两个形状为 `(1000,)` 的 FP32 张量 `x` 和 `y`，`x` 的值全为 2.0，`y` 的值全为 3.0
2. 将它们搬到 NPU
3. 在 NPU 上计算 `z = x² + y`（提示：可以用 `torch.pow(x, 2)` 或 `x * x`）
4. 将结果搬回 CPU 并验证所有元素等于 7.0

**预期结果**：`z` 的所有元素均为 $2.0^2 + 3.0 = 7.0$

**API 速查**：

| 操作 | API | 示例 |
|:---|:---|:---|
| 创建全量张量 | `torch.full(shape, value, dtype=...)` | `torch.full((1000,), 2.0, dtype=torch.float32)` |
| 搬到 NPU | `tensor.npu()` | `x_npu = x.npu()` |
| 搬回 CPU | `tensor.cpu()` | `z = z_npu.cpu()` |
| 逐元素平方 | `x * x` 或 `torch.pow(x, 2)` | `sq = x_npu * x_npu` |
| 逐元素加法 | `a + b` | `z_npu = sq + y_npu` |
| 验证相等 | `torch.allclose(a, b)` | `torch.allclose(z, expected)` |

In [ ]:
import torch
import torch_npu

# ========== 第 1 题：在 NPU 上计算 z = x² + y ==========
# TODO: 在下方完成你的代码

# Step 1: 创建 x 和 y（CPU 上，FP32，形状 (1000,)，x 全为 2.0，y 全为 3.0）
x = 
y = 

# Step 2: 搬到 NPU
x_npu = 
y_npu = 

# Step 3: 在 NPU 上计算 z = x² + y
z_npu = 

# Step 4: 搬回 CPU
z = 

# 验证
expected = torch.full((1000,), 7.0)
print(f"z 的前 5 个元素: {z[:5].tolist()}")
print(f"z 的设备: {z_npu.device}")
print(f"验证结果: {'✓ 通过' if torch.allclose(z, expected) else '✗ 失败'}")

### 第 2 题（进阶 ★★☆）：CPU vs NPU 矩阵乘法性能对比

**要求**：
1. 对矩阵规模 `N = [128, 512, 2048]`，分别创建两个 `N×N` 的随机 FP32 矩阵
2. 分别在 CPU 和 NPU 上执行矩阵乘法，用 `time.time()` 计时
3. 验证 CPU 和 NPU 结果一致（`torch.allclose`，容差 `atol=1e-1`）
4. 打印每种规模下 CPU 和 NPU 的耗时及加速比

**API 速查**：

| 操作 | API | 示例 |
|:---|:---|:---|
| 创建随机矩阵 | `torch.randn(N, N, dtype=...)` | `a = torch.randn(512, 512, dtype=torch.float32)` |
| 搬到 NPU | `tensor.npu()` | `a_npu = a.npu()` |
| 矩阵乘法 | `torch.matmul(a, b)` 或 `a @ b` | `c = torch.matmul(a_npu, b_npu)` |
| NPU 同步 | `torch.npu.synchronize()` | `torch.npu.synchronize()` |
| 搬回 CPU | `tensor.cpu()` | `c_cpu = c_npu.cpu()` |
| 验证（带容差） | `torch.allclose(a, b, atol=...)` | `torch.allclose(c_cpu, c_npu.cpu(), atol=1e-1)` |
| 计时 | `time.time()` | `t0 = time.time(); ...; elapsed = time.time() - t0` |

**提示**：
- NPU 计算是异步的，计时前需要 `torch.npu.synchronize()` 确保计算完成
- 注意：NPU 首次执行会有编译开销，如果想测纯计算性能可以先做一次 warmup

In [ ]:
import time

# ========== 第 2 题：CPU vs NPU 矩阵乘法性能对比 ==========
# TODO: 在下方完成你的代码

sizes = [128, 512, 2048]

print(f"{'规模':>8} | {'CPU 耗时':>12} | {'NPU 耗时':>12} | {'加速比':>8} | {'结果一致':>8}")
print("-" * 65)

for N in sizes:
    # Step 1: 创建两个 N×N 随机矩阵（CPU 上）
    a_cpu = 
    b_cpu = 

    # Step 2: CPU 矩阵乘法计时
    

    # Step 3: NPU 矩阵乘法计时（别忘了 synchronize）
    

    # Step 4: 验证结果一致
    

    # 打印结果
    

### 第 3 题（挑战 ★★★）：Batch 矩阵乘法 vs 循环单次矩阵乘法

**背景**：通过 `torch_npu` 调用的每个 API 都对应一个 CANN 算子，即一次 NPU Kernel 启动。`torch.bmm` 一次调用完成 B 个矩阵乘法，而循环 `torch.matmul` 需要 B 次独立调用——每次调用都有 Kernel 启动开销和 Python 调度开销。随着 B 增大，差异会越来越显著。

**要求**：
1. 对 batch 大小 `B = [8, 32, 128]`，创建形状为 `(B, 256, 256)` 的随机 FP32 矩阵对，搬到 NPU
2. **Batch 方式**：用 `torch.bmm(a, b)` 一次完成 B 个矩阵乘法，计时
3. **循环方式**：用 `for i in range(B): torch.matmul(a[i], b[i])` 逐个计算，计时
4. 验证两种方式结果一致（`torch.allclose`，`atol=1e-1`）
5. 打印每种 batch 大小下两种方式的耗时及加速比，观察 B 增大时差异如何变化

**思考题**（选做）：
- 为什么 B 越大，循环方式越慢？开销来自哪里？
- 什么情况下循环方式的劣势不明显？（提示：矩阵本身很大时）
- 在模型开发中，如何避免不必要的循环调用？

> 📌 **后续课程将深入解释**：Kernel 启动开销的底层原因、NPU 任务调度器（TS）的工作机制、以及如何通过算子融合进一步减少调用次数，将在 Ascend C 算子开发系列课程中详细讲解。

**API 速查**：

| 操作 | API | 示例 |
|:---|:---|:---|
| 创建 batch 张量并搬到 NPU | `torch.randn(B, M, K, dtype=...).npu()` | `a = torch.randn(32, 256, 256, dtype=torch.float32).npu()` |
| Batch 矩阵乘法 | `torch.bmm(a, b)` | `c = torch.bmm(a, b)`  # a:(B,M,K) b:(B,K,N) → c:(B,M,N) |
| 单次矩阵乘法 | `torch.matmul(a[i], b[i])` | `c_i = torch.matmul(a[0], b[0])` |
| NPU 同步 | `torch.npu.synchronize()` | `torch.npu.synchronize()` |
| 验证（带容差） | `torch.allclose(a, b, atol=...)` | `torch.allclose(c_bmm, c_loop, atol=1e-1)` |
| 计时 | `time.time()` | `t0 = time.time(); ...; elapsed = (time.time()-t0)/reps*1000` |

In [ ]:
# ========== 第 3 题：Batch 矩阵乘法 vs 循环单次矩阵乘法 ==========
# TODO: 在下方完成你的代码

batch_sizes = [8, 32, 128]
M, K, N = 256, 256, 256
reps = 20

print(f"{'B':>6} | {'bmm 耗时':>12} | {'循环 耗时':>12} | {'加速比':>8} | {'结果一致':>8}")
print("-" * 65)

for B in batch_sizes:
    # Step 1: 创建 (B, M, K) 和 (B, K, N) 随机矩阵，搬到 NPU
    

    # Step 2: Warmup
    

    # Step 3: torch.bmm 计时
    

    # Step 4: 循环 torch.matmul 计时
    

    # Step 5: 验证结果一致
    

    # Step 6: 打印结果
    

---
## 五、验证与答案

完成全部三道实践题后，运行下方代码查看参考答案与批改结果。

In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd() / 'answer',
    Path.cwd() / 'quick_start' / 'cann_basics' / 'answer',
):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find quick_start/cann_basics/answer')
from grade_05 import grade
grade(globals())

---
## 六、下一步学习建议

完成本课程后，你已经具备了 CANN 和昇腾 NPU 的基础认知。接下来推荐以下方向：

| 方向 | 链接 | 适合人群 |
|:---|:---|:---|
| 体验自定义算子开发 | [第一个自定义算子](../first_custom_operator/first_custom_operator.ipynb) | 想了解算子是怎么写的 |
| 体验算子 API 调用 | [第一个算子 API 调用](../first_operator_api_call/first_operator_api_call.ipynb) | 想了解如何调用 CANN 内置算子 |
| 系统学习算子开发 | [Ascend C 算子开发系列](../../tutorials/ascendc_operator_development_light) | 想掌握底层编程能力 |
| 大模型推理实战 | [Qwen3-8B 推理](../../tutorials/llm_inference/qwen3_8b/02_baseline_inference.ipynb) | 想跑通真实大模型 |

> 💡 **学习路径建议**：先体验`第一个自定义算子`感知算子开发全流程 → 再系统学习 Ascend C 算子开发系列 → 最后结合大模型推理/训练深入实践。